In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem,DataStructs
from rdkit.Chem import  Descriptors, Crippen, Lipinski, MolSurf
from rdkit.Chem import rdFingerprintGenerator
from propy.PyPro import GetProDes
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import math
pd.set_option('display.max_colwidth', None)

C:\Users\blons\code\trial\envs\chem_env\Lib\site-packages\pyopsin\pyopsin.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


# Drug properties

In [2]:
def drug_properties(smiles):

    smiles = str(smiles)
        
    if "|" in smiles:
        smiles = smiles.split("|")[0].strip()
    
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        return False
    #Bond features

    single = 0
    double = 0
    triple = 0
    aromatic = 0
    conjugated = 0
    total_bonds = mol.GetNumBonds()
    ringBonds = 0
    
    for bond in mol.GetBonds():
        if bond.IsInRing():
            ringBonds +=1
        btype = bond.GetBondType()
        if btype == Chem.rdchem.BondType.SINGLE:
            single += 1
        elif btype == Chem.rdchem.BondType.DOUBLE:
            double += 1
        elif btype == Chem.rdchem.BondType.TRIPLE:
            triple += 1
        elif btype == Chem.rdchem.BondType.AROMATIC:
            aromatic += 1
        
        if bond.GetIsConjugated():
            conjugated += 1

    
   #Hybridization
    
    SP3 = 0
    SP2 = 0
    SP = 0
    
    for i in mol.GetAtoms():
        hybridization = i.GetHybridization()
        if hybridization == Chem.rdchem.HybridizationType.SP3:
            SP3 +=1
        elif hybridization == Chem.rdchem.HybridizationType.SP2:
            SP2 +=1
        elif hybridization == Chem.rdchem.HybridizationType.SP:
            SP +=1


    #formalCharge
    
    overall_charge = 0
    for i in mol.GetAtoms():
        overall_charge += i.GetFormalCharge()


    #Aromatic Cycles
    heteroAromatic = Lipinski.NumAromaticHeterocycles(mol)
    carboAromatic = Lipinski.NumAromaticCarbocycles(mol)

    
    #TPSA
    
    Tpsa = MolSurf.TPSA(mol)


    #Hydrogen Bonds
    
    Hdonors = Lipinski.NumHDonors(mol)
    Hacceptors = Lipinski.NumHAcceptors(mol)

    
    #logP
    
    logp = Crippen.MolLogP(mol)

    
    #Mr
    
    weight = Descriptors.ExactMolWt(mol)

    
    #Morgan fingerprint
    morgan_fp_gen = rdFingerprintGenerator.GetMorganGenerator(includeChirality=True, radius=2, fpSize=2048,countSimulation=False)
    
    fp = morgan_fp_gen.GetFingerprint(mol)
    arr = np.zeros((2048,), dtype=int)
    
    DataStructs.ConvertToNumpyArray(fp, arr)
    
    output = {"Molecular weight": weight,
             "LogP": logp,
             "Hydrogen bond Donors": Hdonors,
             "Hydrogen bond Acceptors": Hacceptors,
             "TPSA": Tpsa,
             "Aromatic Heterocycles": heteroAromatic,
             "Aromatic Carbocycles": carboAromatic,
             "Overall charge": overall_charge,
             'SP3' : SP3,
             'SP2' : SP2,
             'SP' : SP,
             "Single bonds": single,
             "Double bonds": double,
             "Triple bonds": triple,
             "Aromatic bonds": aromatic,
             "Rotatable bonds": Lipinski.NumRotatableBonds(mol),
             "Conjugated bonds": conjugated,
             "Conjugation ratio": conjugated / total_bonds if total_bonds else 0,
             "Bonds in ring": ringBonds
            }
    for i, bit in enumerate(arr):
        output[f"drug_FP_{i}"] = int(bit)
    return output

# Protein properties

In [3]:
def protein_properties(sequence):

    sequence = sequence.upper()
    
    allowed = set("ARNDCEQGHILKMFPSTWYV")
     
    if set(sequence) - allowed:
        return False
 
    des = GetProDes(sequence)
    analysis = ProteinAnalysis(sequence)
  
    protein_features = {}

    
    AAC = des.GetAAComp()
    DPC = des.GetDPComp()
    CTD = des.GetCTD()
    PAAC = des.GetPAAC(lamda=10, weight=0.05)
    APAAC = des.GetAPAAC(lamda=10, weight=0.05)
    moreau_broto = des.GetMoreauBrotoAuto()
    moran = des.GetMoranAuto()
    geary = des.GetGearyAuto()
    Mr = analysis.molecular_weight()
    aromaticity = analysis.aromaticity()
    IP = analysis.isoelectric_point()
    II = analysis.instability_index()
    GRAVY = analysis.gravy()

    protein_features.update(AAC)
    protein_features.update(DPC)
    protein_features.update(CTD)
    protein_features.update(PAAC)
    protein_features.update(APAAC)
    protein_features.update(moreau_broto)
    protein_features.update(moran)
    protein_features.update(geary)
    protein_features.update({"Mr" : Mr,
                            "Aromaticity" : aromaticity,
                            "Isoelectric point" : IP,
                            "Instability index" : II,
                            "GRAVY" : GRAVY})
 
    return protein_features

# Applying the properties

In [10]:
trial_num = int(input("Enter trial number: "))
# chunk_num = int(input("Enter chunk number: "))
    
protein_cache = {}
drug_cache = {}

for i in range(10):
    
    all_properties = []
    
    chunk = pd.read_csv(f"chunks/chunk_{i+21}_{trial_num}.csv")
    
    print(chunk.shape)
    print(chunk.columns)
    
    print('\n', chunk.iloc[:,3].nunique())
    print(chunk.iloc[:,1].nunique())
    print(len(chunk))
        
    unique_sequences = chunk.iloc[:, 3].dropna().unique()
    unique_SMILES = chunk.iloc[:, 1].dropna().unique()
    
    for seq in unique_sequences:
    
        seq = str(seq).upper().strip()
    
        if len(seq) < 11:
            continue
            
        if seq not in protein_cache:
            intermidiate = protein_properties(seq)
    
            if intermidiate != False:
                protein_cache[seq] = intermidiate
            
            else:
                print('Incorrect sequence!\n', seq)
     
    for smiles in unique_SMILES:
    
        if smiles not in drug_cache:
            intermidiate = drug_properties(smiles)
    
            if intermidiate != False:
                drug_cache[smiles] = intermidiate
                
            else:
                print('Incorrect SMILES!\n', smiles)
        
    for row in chunk.itertuples(index=False):
    
        censored = 0
        smiles = row[1]
        sequence = str(row[3]).upper().strip()
        ki = row[2]
        
        censored = 0
        
        if ">" in ki or "<" in ki or "=" in ki:
        
            ki = (
                ki.replace("<", "")
                  .replace(">", "")
                  .replace("=", "")
            )
        
            censored = 1
        
        ki = float(ki)
    
        if ki == 0.0:
            pki = 0.0
        
        else:
            pki = 9 - math.log10(ki)
    
        if smiles in drug_cache and sequence in protein_cache:
            
            properties = drug_cache[smiles].copy()
            properties.update(protein_cache[sequence])
            properties["pKi"] = pki
            properties["Inaccurate"] = censored
            
            all_properties.append(properties)
    
    df = pd.DataFrame(all_properties)
    df.to_csv(f"Dataframes/chunk_{i+21}_{trial_num}_dataframe.csv", index=False)
    
    print("Chunk processed!")

Enter trial number:  1


(29617, 4)
Index(['Unnamed: 0', 'Ligand SMILES', 'Ki (nM)',
       'BindingDB Target Chain Sequence 1'],
      dtype='str')

 764
15350
29617
Chunk processed!
(20272, 4)
Index(['Unnamed: 0', 'Ligand SMILES', 'Ki (nM)',
       'BindingDB Target Chain Sequence 1'],
      dtype='str')

 645
10352
20272
Chunk processed!
(38065, 4)
Index(['Unnamed: 0', 'Ligand SMILES', 'Ki (nM)',
       'BindingDB Target Chain Sequence 1'],
      dtype='str')

 708
16489
38065
Chunk processed!
(27072, 4)
Index(['Unnamed: 0', 'Ligand SMILES', 'Ki (nM)',
       'BindingDB Target Chain Sequence 1'],
      dtype='str')

 727
12241
27072
Chunk processed!
(26691, 4)
Index(['Unnamed: 0', 'Ligand SMILES', 'Ki (nM)',
       'BindingDB Target Chain Sequence 1'],
      dtype='str')

 804
14249
26691
Chunk processed!
(18055, 4)
Index(['Unnamed: 0', 'Ligand SMILES', 'Ki (nM)',
       'BindingDB Target Chain Sequence 1'],
      dtype='str')

 725
9083
18055
Chunk processed!
(27513, 4)
Index(['Unnamed: 0', 'Ligand SMILES'

In [2]:
final_df = pd.DataFrame([])
trial_num = int(input("Enter the trial number:"))

for i in range(31):
    
    df = pd.read_csv(f"Dataframes/chunk_{i}_{trial_num}_dataframe.csv")

    print(i)
    
    final_df = pd.concat([final_df, df], ignore_index = True)

Enter the trial number: 1


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30


In [3]:
final_df.to_parquet("training_data.parquet")

In [4]:
final_df

,Molecular weight,LogP,Hydrogen bond Donors,Hydrogen bond Acceptors,TPSA,Aromatic Heterocycles,Aromatic Carbocycles,Overall charge,SP3,SP2,...,GearyAuto_Mutability28,GearyAuto_Mutability29,GearyAuto_Mutability30,Mr,Aromaticity,Isoelectric point,Instability index,GRAVY,pKi,Inaccurate
0,554.299202,3.9584,4,5,138.61,0,2,0,18,22,...,0.859,1.023,1.014,10792.6645,0.050505,8.830766,47.528283,0.209091,9.619789,0
1,538.269239,3.5414,4,5,121.37,2,2,0,10,30,...,0.859,1.023,1.014,10792.6645,0.050505,8.830766,47.528283,0.209091,9.602060,0
2,486.263091,3.5149,3,4,92.69,1,2,0,13,23,...,0.859,1.023,1.014,10792.6645,0.050505,8.830766,47.528283,0.209091,9.387216,0
3,480.298808,3.6311,3,4,84.24,0,2,0,19,16,...,0.859,1.023,1.014,10792.6645,0.050505,8.830766,47.528283,0.209091,9.096910,0
4,466.283158,3.2410,3,4,84.24,0,2,0,18,16,...,0.859,1.023,1.014,10792.6645,0.050505,8.830766,47.528283,0.209091,9.004365,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
602788,790.357779,5.9913,5,11,171.68,1,4,0,19,39,...,0.940,1.041,1.049,46458.1501,0.113801,6.591868,41.338039,0.169976,10.599998,0
602789,376.152203,0.4647,4,7,108.61,0,2,0,13,14,...,1.029,1.006,1.032,73497.2109,0.114458,7.819193,40.476054,0.563253,6.149967,0
602790,448.173332,1.6455,3,9,123.91,0,2,0,14,18,...,1.029,1.006,1.032,73497.2109,0.114458,7.819193,40.476054,0.563253,5.122629,0
602791,376.152203,0.4647,4,7,108.61,0,2,0,13,14,...,1.056,1.034,1.033,72895.9388,0.110119,7.451387,44.717872,0.584524,8.619789,0
